# 10 · nano-GPT 手撸（中文字符级）

> **学习目标**：从零写出一个能跑的 char-level GPT，在唐诗语料上**训练 + 生成**，亲眼看 loss 下降、亲眼看模型从「随机字符」一步步学到「带标点的中文格式」。
>
> **预备**：05、06、07 已过。要看懂 attention 与训练循环。
>
> **为什么重要**：Karpathy 一直强调「You don't really understand it until you build it」。所有现代 LLM（GPT-2/3/4、Llama、Qwen）的核心结构和这个 200 行 demo **本质相同**，差别只在规模与细节。

**两种配置一键切换**：`MINI`（CPU 5 分钟 / GPU 20 秒，验证流程）vs `FULL`（GPU 5–10 分钟，生成质量更好）。第一次跑用 `MINI`。

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math, time
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', device)

## 1. 准备语料 —— 唐诗 30 首

**为什么唐诗**：篇幅短、格式整齐（5/7 字 + 标点）、内嵌结构（押韵 + 对仗），字符级模型能在几百步内学到「这一行的字数」「这里该是逗号还是句号」等浅层规律。

In [ ]:
POEMS = [
    '静夜思 李白\n床前明月光，疑是地上霜。举头望明月，低头思故乡。',
    '春晓 孟浩然\n春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。',
    '登鹳雀楼 王之涣\n白日依山尽，黄河入海流。欲穷千里目，更上一层楼。',
    '望庐山瀑布 李白\n日照香炉生紫烟，遥看瀑布挂前川。飞流直下三千尺，疑是银河落九天。',
    '早发白帝城 李白\n朝辞白帝彩云间，千里江陵一日还。两岸猿声啼不住，轻舟已过万重山。',
    '黄鹤楼送孟浩然之广陵 李白\n故人西辞黄鹤楼，烟花三月下扬州。孤帆远影碧空尽，唯见长江天际流。',
    '鹿柴 王维\n空山不见人，但闻人语响。返景入深林，复照青苔上。',
    '山中送别 王维\n山中相送罢，日暮掩柴扉。春草明年绿，王孙归不归。',
    '杂诗 王维\n君自故乡来，应知故乡事。来日绮窗前，寒梅著花未。',
    '相思 王维\n红豆生南国，春来发几枝。愿君多采撷，此物最相思。',
    '江雪 柳宗元\n千山鸟飞绝，万径人踪灭。孤舟蓑笠翁，独钓寒江雪。',
    '凉州词 王之涣\n黄河远上白云间，一片孤城万仞山。羌笛何须怨杨柳，春风不度玉门关。',
    '出塞 王昌龄\n秦时明月汉时关，万里长征人未还。但使龙城飞将在，不教胡马度阴山。',
    '凉州词 王翰\n葡萄美酒夜光杯，欲饮琵琶马上催。醉卧沙场君莫笑，古来征战几人回。',
    '春夜喜雨 杜甫\n好雨知时节，当春乃发生。随风潜入夜，润物细无声。',
    '绝句 杜甫\n两个黄鹂鸣翠柳，一行白鹭上青天。窗含西岭千秋雪，门泊东吴万里船。',
    '春望 杜甫\n国破山河在，城春草木深。感时花溅泪，恨别鸟惊心。',
    '江南春 杜牧\n千里莺啼绿映红，水村山郭酒旗风。南朝四百八十寺，多少楼台烟雨中。',
    '山行 杜牧\n远上寒山石径斜，白云生处有人家。停车坐爱枫林晚，霜叶红于二月花。',
    '清明 杜牧\n清明时节雨纷纷，路上行人欲断魂。借问酒家何处有，牧童遥指杏花村。',
    '夜雨寄北 李商隐\n君问归期未有期，巴山夜雨涨秋池。何当共剪西窗烛，却话巴山夜雨时。',
    '嫦娥 李商隐\n云母屏风烛影深，长河渐落晓星沉。嫦娥应悔偷灵药，碧海青天夜夜心。',
    '渭城曲 王维\n渭城朝雨浥轻尘，客舍青青柳色新。劝君更尽一杯酒，西出阳关无故人。',
    '九月九日忆山东兄弟 王维\n独在异乡为异客，每逢佳节倍思亲。遥知兄弟登高处，遍插茱萸少一人。',
    '枫桥夜泊 张继\n月落乌啼霜满天，江枫渔火对愁眠。姑苏城外寒山寺，夜半钟声到客船。',
    '望天门山 李白\n天门中断楚江开，碧水东流至此回。两岸青山相对出，孤帆一片日边来。',
    '赋得古原草送别 白居易\n离离原上草，一岁一枯荣。野火烧不尽，春风吹又生。远芳侵古道，晴翠接荒城。又送王孙去，萋萋满别情。',
    '登高 杜甫\n风急天高猿啸哀，渚清沙白鸟飞回。无边落木萧萧下，不尽长江滚滚来。万里悲秋常作客，百年多病独登台。艰难苦恨繁霜鬓，潦倒新停浊酒杯。',
    '回乡偶书 贺知章\n少小离家老大回，乡音无改鬓毛衰。儿童相见不相识，笑问客从何处来。',
    '咏柳 贺知章\n碧玉妆成一树高，万条垂下绿丝绦。不知细叶谁裁出，二月春风似剪刀。',
]
CORPUS = '\n\n'.join(POEMS)
print(f'语料：{len(POEMS)} 首，共 {len(CORPUS)} 字')
print('---前 100 字预览---')
print(CORPUS[:100])

In [ ]:
# 字符级 vocab（最简实现：所有不同字符 + 标点）
chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    return ''.join(itos[i] for i in ids)

print(f'vocab_size = {vocab_size}（含中文字 + 标点 + 换行 + 空格）')
print('前 20 个 vocab:', chars[:20])

# 整个语料转成一个张量
data = torch.tensor(encode(CORPUS), dtype=torch.long)
print('data shape:', data.shape, '  dtype:', data.dtype)

# 90/10 train/val 切分
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f'train: {len(train_data)} tokens   val: {len(val_data)} tokens')

## 2. 训练数据准备 —— 随机采样窗口

**思路**：从语料里随机选 `B` 个起点，每个起点抽一段长度 `block_size` 的 token 作为输入 `x`，对应「向右挪 1 位」的同长度切片作为目标 `y`。

`y[t]` = `x` 前 `t+1` 个 token 的「下一个真值」。loss 是按位置算交叉熵的平均。

In [ ]:
def get_batch(split: str, block_size: int, batch_size: int):
    src = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i: i + block_size]       for i in ix])   # (B, L)
    y = torch.stack([src[i + 1: i + 1 + block_size] for i in ix])  # (B, L)
    return x.to(device), y.to(device)

# 演示
xb, yb = get_batch('train', block_size=16, batch_size=2)
print('x[0]:', decode(xb[0].tolist()))
print('y[0]:', decode(yb[0].tolist()), '  <- 比 x[0] 向右挪 1 位')

## 3. 模型 —— GPT 三件套：Block / FFN / Head

**结构**（Pre-LayerNorm，现代主流）：
```
Block:
  x = x + Attention(LN(x))
  x = x + FFN(LN(x))
```

**与 notebook 07 的差别**：07 写的是无 mask 的双向 attention，这里要加 **causal mask**（自回归 LM 不许看未来）。

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        # causal mask: 下三角是 1（保留），上三角是 0（屏蔽）
        self.register_buffer('mask',
                             torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                      # 3 x (B, L, D)
        q = q.view(B, L, self.n_head, self.d_k).transpose(1, 2)      # (B, H, L, dk)
        k = k.view(B, L, self.n_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.n_head, self.d_k).transpose(1, 2)
        att = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)          # (B, H, L, L)
        att = att.masked_fill(self.mask[:, :, :L, :L] == 0, float('-inf'))
        att = att.softmax(-1)
        att = self.drop(att)
        y = att @ v                                                   # (B, H, L, dk)
        y = y.transpose(1, 2).contiguous().view(B, L, D)
        return self.proj(y)

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [ ]:
class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)        # 可学习位置编码（GPT-2 风格）
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # weight tying：把 lm_head 与 tok_emb 共享权重，省 vocab*d 参数
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, L = idx.shape
        pos = torch.arange(L, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                # (B, L, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0, top_k: int | None = None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]                # 截断到 block_size 之内
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature             # 只看最后一个位置
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = logits.softmax(-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

## 4. 训练配置 —— MINI / FULL 一键切换

**MINI**：500 iters / 模型 ~50K 参 / 1 分钟跑完。**目标只是 loss 单调下降、不爆 nan**。
**FULL**：3000 iters / 模型 ~500K 参 / 在 GPU 上 5-10 分钟。**生成结果开始有诗的样子**。

把下面 `MODE` 改成 `'FULL'` 切换。

In [ ]:
MODE = 'MINI'   # 'MINI' or 'FULL'

CFG = {
    'MINI': dict(d_model=64,  n_head=2, n_layer=2, block_size=64,
                  batch_size=16, iters=500,  lr=3e-3, dropout=0.0,
                  eval_interval=100, eval_iters=20),
    'FULL': dict(d_model=192, n_head=4, n_layer=4, block_size=128,
                  batch_size=32, iters=3000, lr=3e-4, dropout=0.1,
                  eval_interval=300, eval_iters=50),
}[MODE]
print(f'MODE = {MODE}')
for k, v in CFG.items():
    print(f'  {k}: {v}')

In [ ]:
model = NanoGPT(
    vocab_size=vocab_size,
    d_model=CFG['d_model'],
    n_head=CFG['n_head'],
    n_layer=CFG['n_layer'],
    block_size=CFG['block_size'],
    dropout=CFG['dropout'],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'参数量: {n_params:,} ({n_params/1e6:.3f} M)')
print(f'tied weights 已生效: {model.lm_head.weight is model.tok_emb.weight}')

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(CFG['eval_iters'])
        for k in range(CFG['eval_iters']):
            x, y = get_batch(split, CFG['block_size'], CFG['batch_size'])
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])
history = {'iter': [], 'train': [], 'val': []}

t0 = time.perf_counter()
model.train()
for it in range(CFG['iters']):
    if it % CFG['eval_interval'] == 0 or it == CFG['iters'] - 1:
        losses = estimate_loss()
        history['iter'].append(it); history['train'].append(losses['train']); history['val'].append(losses['val'])
        print(f'iter {it:5d}: train {losses["train"]:.3f}   val {losses["val"]:.3f}')

    x, y = get_batch('train', CFG['block_size'], CFG['batch_size'])
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

print(f'\n训练总时间: {time.perf_counter() - t0:.1f}s')

plt.figure(figsize=(7, 3))
plt.plot(history['iter'], history['train'], '-o', label='train')
plt.plot(history['iter'], history['val'],   '-o', label='val')
plt.xlabel('iter'); plt.ylabel('loss'); plt.legend(); plt.grid(True)
plt.title(f'NanoGPT loss ({MODE})')
plt.show()

## 5. 生成 —— 看模型学到了什么

**MINI** 配置下，500 步训练通常只能让模型学到：
- 中文字符比拉丁字符多
- 标点更倾向出现在某些位置
- 但还不会形成像样的诗

**FULL** 配置下：能出现像样的「7 字 + 逗号 + 7 字 + 句号」结构，偶尔有押韵。

**这就是 char-level 模型的上限**。要更好的中文生成必须上 BPE tokenizer + 更大模型 + 更多数据。

In [ ]:
def sample(prompt: str = '春', max_new_tokens: int = 80, temperature: float = 0.9, top_k: int = 10):
    model.eval()
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(out[0].tolist())

for prompt in ['春', '月', '山中', '夜']:
    print(f'--- 起始字: {prompt!r} ---')
    print(sample(prompt))
    print()

## 深入思考

1. **为什么用 Pre-LN 而不是原始 Transformer 的 Post-LN？**
   - Pre-LN 训练更稳，深层网络不需要复杂的 warmup。GPT-2 起所有现代 LLM 都用 Pre-LN。
2. **weight tying 把 `lm_head.weight` 与 `tok_emb.weight` 共享，省了什么？**
   - 省了 `vocab × d_model` 个参数。对大词表（50k+）的真实 LLM 是非常显著的节省。GPT-2、Qwen 都用了。
3. **MINI 配置参数量这么小，为什么仍然能学？**
   - 任务简单（vocab 仅几百，模式是规整唐诗）。规模与任务难度要匹配——小任务小模型反而更稳。
4. **`generate` 里为什么要 `idx[:, -block_size:]`？**
   - 模型只在固定 `block_size` 窗口上训练。生成超过这个长度时，截断到最后一段。这就是「上下文窗口」的真实含义。
5. **能不能扩到长上下文？**
   - 不能直接扩——位置 embedding 表大小是 `block_size`，超出就 OOR。要么用相对位置/RoPE（见 13/14 号 notebook），要么训练时就用大 block_size。

**改一改**：
- 把 `top_k` 改成 1，对比是不是变得很重复（greedy 容易陷死循环）
- 把 `temperature` 改成 0.3，看生成是不是变「保守」
- 把 `MODE='FULL'` 跑一次，看 loss 能不能降到 2 以下、生成质量是否变化

## 自检 ✅

- [ ] 在白板上画出 nano-GPT 的 5 层结构（emb → drop → N×Block → LN → lm_head）。
- [ ] 解释「causal mask 是怎么阻止模型看未来的」（softmax 把 -inf 变 0）。
- [ ] 解释「Pre-LN 与 Post-LN 的差别」。
- [ ] 解释「weight tying 省什么参数」。
- [ ] 现场写出 `get_batch` 的 5 行核心。
- [ ] 对比 `top_k=1` vs `top_k=10`、`temperature=0.3` vs `1.5` 的生成差别。

## 下一步

→ [`11_tokenizer_compare.ipynb`](11_tokenizer_compare.ipynb)